# Retrieval evaluation — precision@k + automatic diagnostics (RQ1 / RQ4)

`26.06.md` names **precision** as *the* evaluation of the retrieval tool (recall is not
computable — the true relevant set can't be enumerated). This notebook makes that claim
measurable over the **extended multilingual index** (ECHR EN + RIS AT + Swiss CH):

1. a **frozen Bucket-1 query set** (24 queries, EN + DE, incl. the Austrian and Swiss
   registers),
2. **automatic diagnostics** that need no hand labels:
   - jurisdiction / language mix of the top-k (does a German query reach the German corpora?
     — the honest RQ4 retrieval-generality measurement),
   - ECHR **section composition** (LAW vs FACTS — quantifies the known \"recitals instead
     of reasoning\" problem),
   - duplicate-document rate, genre mix, cosine stats,
3. a **judgment template** → `data/retrieval_judgments_template.csv` (one row per hit,
   blank `relevant` column). Fill it (1/0), save as `data/retrieval_judgments.csv`, re-run
   → **precision@k overall / per language / per jurisdiction**. Degrades gracefully: without
   judgments only the automatic part runs.

Reuses the deployed pipeline by exec'ing the cells of `rag_echr_ris.ipynb` (same config,
same cache, same retrieval code — nothing re-implemented, nothing re-embedded).

## 1. Load the deployed RAG pipeline (exec the notebook's cells, cache-hot)

In [ ]:
import json
from pathlib import Path

DATA_DIR   = Path("../data")
REPORT_DIR = Path("../reports")
RAG_NB     = Path("rag_echr_ris.ipynb")
TEMPLATE_OUT  = DATA_DIR / "retrieval_judgments_template.csv"
JUDGMENTS_IN  = DATA_DIR / "retrieval_judgments.csv"      # <- you save the filled template here
REPORT_OUT    = REPORT_DIR / "retrieval_evaluation_report.md"
EVAL_K = 6

# exec the pipeline cells up to and including retrieve() — identified by marker strings so
# this stays in sync with the deployed notebook, not a copy of it.
MARKERS = ["DATA_DIR  = Path", "def load_json_records", "ECHR_ANCHORS = [",
           "reusable ECHR genre helper", "records, chunks = [], []",
           "_embedder = None", "def _to_hit"]
nb = json.loads(RAG_NB.read_text())
ns_src = []
for c in nb["cells"]:
    if c["cell_type"] != "code":
        continue
    src = "".join(c["source"])
    if any(m in src for m in MARKERS):
        ns_src.append(src)
print(f"loaded {len(ns_src)} pipeline cells from {RAG_NB.name}")
for src in ns_src:
    exec(src)
assert index is not None, "index not built — run rag_echr_ris.ipynb first (embedding cache)"
print(f"pipeline ready: {index.ntotal} vectors")

loaded 7 pipeline cells from rag_echr_ris.ipynb
inputs: ['echr_parental_alienation.json', 'ris_parental_alienation.json', 'swiss_parental_alienation.json']
chunk=600w/80o | ECHR skip={'OPINION', 'RELEVANT_LAW'} law_only=False | dev_cap=None
RIS decisions=True (civil only) | Swiss=True
genre routing: exclude={'communicated'} | MMR fetch_k=30 lambda=0.7
normalisers ready: ['echr', 'ris', 'swiss']
section splitter + chunker ready
genre helper ready: ('merits', 'admissibility', 'communicated', 'other') | low-info: {'communicated'}
  echr : 1116 records from echr_parental_alienation.json
  ris : 38 principles + 479 civil decisions (dropped 31 criminal-senate/AUSL 'Text' records — Entfremdung homonym / ECtHR summaries)
  ris  : 517 records from ris_parental_alienation.json
  swiss: dropped 24 Rechenschaftsbericht records (court annual reports, not case law — multi-case digests that flood the top-k)
  swiss: 2007 records from swiss_parental_alienation.json

ECHR dates: 1114/1116 have YYYY-MM-

## 2. Frozen Bucket-1 query set (24 queries)
Phrasing follows each court's register (ECHR: positive obligations / contact; AT: Obsorge,
Kontaktrecht, Vollzugsma\u00dfnahme; CH: Obhut, Besuchsrecht). All are Bucket-1 questions —
the router evaluation covers the buckets; this set measures *ranking quality*.

In [ ]:
EVAL_QUERIES = [
    # -- EN (ECHR register) --
    ("q01", "en", "positive obligations of the State to maintain contact between parent and child"),
    ("q02", "en", "enforcement of contact rights where one parent obstructs the relationship with the other parent"),
    ("q03", "en", "transfer of custody due to parental alienation"),
    ("q04", "en", "the child's refusal of contact and the weight of the child's wishes"),
    ("q05", "en", "coercive measures to enforce contact orders against the resident parent"),
    ("q06", "en", "passage of time as a decisive factor in reunification of parent and child"),
    ("q07", "en", "role of psychological expert evidence in contact disputes"),
    ("q08", "en", "supervised contact as a measure to rebuild the parent-child relationship"),
    # -- DE (Austrian register) --
    ("q09", "de", "Wann ist von einer Vollzugsma\u00dfnahme abzusehen, wenn sie dem Kindeswohl widerspricht?"),
    ("q10", "de", "Voraussetzungen f\u00fcr gemeinsame Obsorge bei fehlender Kommunikationsbasis der Eltern"),
    ("q11", "de", "Einschr\u00e4nkung des Kontaktrechts wegen Kindeswohlgef\u00e4hrdung"),
    ("q12", "de", "Loyalit\u00e4tskonflikt des Kindes und Beeinflussung durch einen Elternteil"),
    ("q13", "de", "Durchsetzung des Kontaktrechts gegen den Willen des Kindes"),
    ("q14", "de", "Bindungstoleranz als Kriterium f\u00fcr die Obsorgezuteilung"),
    ("q15", "de", "Entziehung der Obsorge wegen Entfremdung des Kindes vom anderen Elternteil"),
    ("q16", "de", "Wechselmodell und Kindeswohl bei strittigen Eltern"),
    # -- DE (Swiss register) --
    ("q17", "de", "Unter welchen Voraussetzungen kann einem Elternteil die Obhut entzogen werden?"),
    ("q18", "de", "Regelung des Besuchsrechts, wenn das Kind den Kontakt ablehnt"),
    ("q19", "de", "begleitetes Besuchsrecht zur Wiederherstellung der Eltern-Kind-Beziehung"),
    ("q20", "de", "Kindesanh\u00f6rung im Verfahren \u00fcber die elterliche Sorge"),
    ("q21", "de", "Verlegung des Wohnsitzes des Kindes ins Ausland gegen den Willen eines Elternteils"),
    ("q22", "de", "Errichtung einer Beistandschaft zur \u00dcberwachung des pers\u00f6nlichen Verkehrs"),
    ("q23", "de", "Entfremdung zwischen Kind und besuchsberechtigtem Elternteil"),
    ("q24", "de", "Kindeswohlgef\u00e4hrdung durch anhaltenden Elternkonflikt"),
]
print(f"{len(EVAL_QUERIES)} queries frozen")

24 queries frozen


## 3. Run retrieval + automatic diagnostics (no labels needed)

In [ ]:
import pandas as pd
from collections import Counter

rows = []
for qid, lang, q in EVAL_QUERIES:
    hits = retrieve(q, k=EVAL_K)
    for rank, h in enumerate(hits, 1):
        rows.append({
            "qid": qid, "query_lang": lang, "query": q, "rank": rank,
            "cos": round(h["score"], 4), "chunk_id": h["chunk_id"], "doc_id": h["id"],
            "jurisdiction": h["jurisdiction"], "source": h["source"],
            "genre": h.get("genre", ""), "section": h.get("section", ""),
            "date": h.get("date", ""), "title": h["title"], "url": h["url"],
            "snippet": h["snippet"][:400].replace("\n", " "),
        })
hits_df = pd.DataFrame(rows)
print(f"{len(hits_df)} hits ({len(EVAL_QUERIES)} queries x top-{EVAL_K})\n")

print("=== jurisdiction mix of top-k, by query language (RQ4: does DE reach the DE corpora?) ===")
juris_mix = hits_df.groupby(["query_lang", "jurisdiction"]).size().unstack(fill_value=0)
print(juris_mix.to_string())

print("\n=== ECHR section composition (LAW = the Court's reasoning; FACTS = recitals) ===")
echr_hits = hits_df[hits_df.source == "echr"]
if len(echr_hits):
    sec = echr_hits.section.value_counts()
    print(sec.to_string())
    law_share = (echr_hits.section == "LAW").mean()
    print(f"LAW share of ECHR hits: {law_share:.2f}  <- the 'recitals problem' quantified")
else:
    law_share = float("nan")
    print("(no ECHR hits)")

print("\n=== duplicate-document rate (same doc twice in one top-k) ===")
dup = 1 - hits_df.groupby("qid").doc_id.nunique().sum() / len(hits_df)
print(f"duplicate rate: {dup:.3f}")

print("\n=== genre mix ===")
print(hits_df.genre.value_counts().to_string())
print("\n=== cosine stats per query language ===")
print(hits_df.groupby("query_lang").cos.describe()[["mean", "min", "max"]].round(3).to_string())

/Users/maksimsmirnov/Desktop/thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
144 hits (24 queries x top-6)

=== jurisdiction mix of top-k, by query language (RQ4: does DE reach the DE corpora?) ===
jurisdiction  AT (OGH)  CH  ECHR
query_lang                      
de                  33  63     0
en                   0   0    48

=== ECHR section composition (LAW = the Court's reasoning; FACTS = recitals) ===
section
FACTS        28
LAW          11
HEADER        7
PROCEDURE     1
OPERATIVE     1
LAW share of ECHR hits: 0.23  <- the 'recitals problem' quantified

=== duplicate-document rate (same doc twice in one top-k) ===
duplicate rate: 0.000

=== genre mix ===
genre
decision         87
merits           31
admissibility    17
principle         9

=== cosine stats per query languag

## 4. Judgment template out / precision@k in (graceful)
Fill `relevant` with 1 (the passage genuinely addresses the query) or 0, save the file as
`data/retrieval_judgments.csv`, re-run this notebook. Until then only the automatic
diagnostics above are reported.

In [ ]:
tmpl = hits_df[["qid", "query_lang", "query", "rank", "cos", "chunk_id",
                "jurisdiction", "genre", "section", "title", "url", "snippet"]].copy()
tmpl["relevant"] = ""      # <- you fill: 1 = addresses the query, 0 = does not

# never clobber a template that already carries labels (it may be under review)
_existing_labels = False
if TEMPLATE_OUT.exists():
    _prev = pd.read_csv(TEMPLATE_OUT)
    _existing_labels = ("relevant" in _prev.columns
                        and pd.to_numeric(_prev.relevant, errors="coerce").notna().any())
if _existing_labels:
    print(f"template already contains labels — NOT overwritten ({TEMPLATE_OUT.name})")
else:
    tmpl.to_csv(TEMPLATE_OUT, index=False)
    print(f"wrote template: {len(tmpl)} rows -> {TEMPLATE_OUT.name}")

prec = None
if JUDGMENTS_IN.exists():
    j = pd.read_csv(JUDGMENTS_IN)
    j = j[pd.to_numeric(j.relevant, errors="coerce").notna()].copy()
    j["relevant"] = j.relevant.astype(int)
    if len(j):
        prec = {
            "overall": j.relevant.mean(),
            "by_lang": j.groupby("query_lang").relevant.mean().round(3).to_dict(),
            "by_juris": j.groupby("jurisdiction").relevant.mean().round(3).to_dict(),
            "by_echr_section": j[j.jurisdiction == "ECHR"].groupby("section")
                                .relevant.mean().round(3).to_dict(),
            "n": len(j),
        }
        print(f"\nPRECISION@{EVAL_K} (n={prec['n']} judged hits)")
        print(f"  overall        : {prec['overall']:.3f}")
        print(f"  by query lang  : {prec['by_lang']}")
        print(f"  by jurisdiction: {prec['by_juris']}")
        print(f"  ECHR by section: {prec['by_echr_section']}")
else:
    print(f"\nno judgments yet ({JUDGMENTS_IN.name} absent) — automatic diagnostics only.")

template already contains labels — NOT overwritten (retrieval_judgments_template.csv)

PRECISION@6 (n=144 judged hits)
  overall        : 0.451
  by query lang  : {'de': 0.594, 'en': 0.167}
  by jurisdiction: {'AT (OGH)': 0.606, 'CH': 0.587, 'ECHR': 0.167}
  ECHR by section: {'FACTS': 0.0, 'HEADER': 0.0, 'LAW': 0.727, 'OPERATIVE': 0.0, 'PROCEDURE': 0.0}


## 5. Report → `reports/retrieval_evaluation_report.md`

In [ ]:
lines = ["# Retrieval evaluation report (RQ1 ranking quality / RQ4 cross-lingual)\n\n"]
lines.append(f"- index: **{index.ntotal}** chunks (ECHR EN + RIS AT + Swiss CH), "
             f"top-{EVAL_K}, genre-filtered + MMR (deployed settings)\n")
lines.append(f"- query set: **{len(EVAL_QUERIES)}** frozen Bucket-1 queries "
             f"({sum(1 for _, l, _ in EVAL_QUERIES if l == 'de')} DE / "
             f"{sum(1 for _, l, _ in EVAL_QUERIES if l == 'en')} EN)\n\n")
lines.append("## Jurisdiction mix of retrieved hits by query language\n\n```\n"
             + juris_mix.to_string() + "\n```\n\n")
if len(echr_hits):
    lines.append(f"## ECHR section composition\n- LAW share of ECHR hits: **{law_share:.2f}** "
                 "(LAW = the Court's own reasoning; the rest are FACTS/HEADER recitals — the "
                 "known base-rate problem, now quantified)\n\n")
lines.append(f"## Duplicate-document rate in top-k\n- **{dup:.3f}** (MMR active)\n\n")
if prec:
    lines.append(f"## Precision@{EVAL_K} (strict criterion, judged on full chunk text)\n"
                 f"- overall: **{prec['overall']:.3f}** (n={prec['n']})\n"
                 f"- by query language: {prec['by_lang']}\n"
                 f"- by jurisdiction: {prec['by_juris']}\n"
                 f"- ECHR hits by section: {prec['by_echr_section']} — LAW-section hits are "
                 "precise, everything else is noise: the LAW-share problem IS the ECHR "
                 "precision problem.\n\n")
else:
    lines.append(f"## Precision@{EVAL_K}\n- **pending**: fill "
                 f"`{TEMPLATE_OUT.name}` -> save as `{JUDGMENTS_IN.name}` -> re-run.\n\n")
lines.append("## Reading\n"
             "- The jurisdiction-mix table is the honest RQ4 retrieval claim: German queries "
             "retrieving from AT/CH corpora (and English from ECHR) shows the shared "
             "multilingual embedding space routes by content, not by accident of language. "
             "Cross-language hits are not errors per se — e5 is multilingual — but a "
             "German query answered *only* by English chunks would mean the German corpora "
             "add no retrieval value.\n"
             "- Annotation protocol (strict criterion): relevant=1 ONLY if the chunk states "
             "the deciding court's OWN rule/reasoning on the query's legal question; facts "
             "recitals, quoted statutes/domestic law, party submissions, headnotes/captions, "
             "operative paragraphs and procedural narration = 0, regardless of topical "
             "overlap. Judged on the full chunk text, not the truncated CSV snippet. Labels "
             "drafted with LLM assistance under this written criterion and reviewed by the "
             "author — a scoped, reproducible measurement, not an IR benchmark.\n")
REPORT_OUT.write_text("".join(lines), encoding="utf-8")
print("wrote", REPORT_OUT)

wrote ../reports/retrieval_evaluation_report.md
